In [7]:
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict

In [3]:
load_dotenv()


True

In [4]:
model = ChatGroq(model="openai/gpt-oss-120b")

In [5]:
#create state

class LLMState(TypedDict):
    question:str
    answer:str


In [8]:
from langgraph.types import StateT
def llm_qa(state:LLMState) -> LLMState:
    #extarct the question from the state
    question = state['question']

    #form a prompt 
    prompt = f'Answer the following question {question}'

    # ask the question to the LLM
    answer = model.invoke(prompt).content

    #update the answer in the state 
    state['answer'] = answer

    return state

In [10]:
#create graph 

graph = StateGraph(LLMState)

#add nodes 
graph.add_node('llm_qa',llm_qa)

#add edge
graph.add_edge(START,'llm_qa')
graph.add_edge('llm_qa',END)

#compile
workflow = graph.compile()

In [12]:
#excute
intial_state =  {'question':'how far is moon from the earth'}

final_state = workflow.invoke(intial_state)

print(final_state)

{'question': 'how far is moon from the earth', 'answer': 'The Moon orbits the Earth at an average distance of about **384\u202f400\u202fkilometers (≈238\u202f900\u202fmiles)**. Because its orbit is elliptical, the actual distance varies:\n\n| Orbital point | Approximate distance from Earth |\n|---------------|---------------------------------|\n| **Perigee** (closest approach) | ~363\u202f300\u202fkm (≈225\u202f600\u202fmi) |\n| **Apogee** (farthest point)   | ~405\u202f500\u202fkm (≈252\u202f000\u202fmi) |\n| **Average (semi‑major axis)** | ~384\u202f400\u202fkm (≈238\u202f900\u202fmi) |\n\n**Other useful facts**\n\n- **Light‑travel time:** It takes about **1.28\u202fseconds** for light (or radio signals) to travel from the Earth to the Moon.\n- **Travel time for spacecraft:** A typical spacecraft takes roughly 3\u202fdays to reach the Moon, depending on the trajectory and propulsion system.\n- **Variability:** The distance changes slightly over long periods due to gravitational inter